# FinMark Data Pipeline
## Milestone 2 | Group 12

This notebook runs the entire Bronze to Silver to Gold pipeline end to end by calling the three layer scripts in `scripts/`. Each script is self-contained and resolves its own file paths using `Path(__file__)`, so this notebook can be run from anywhere inside the project without path errors.

**Pipeline order:**
1. `scripts/profile_bronze_layer.py` - profiles raw CSVs, writes `data/bronze/bronze_quality_report.csv`
2. `scripts/clean_silver_layer.py` - cleans raw CSVs, writes 5 files to `data/silver/`
3. `scripts/build_gold_layer.py` - builds dashboard tables, writes 6 files to `data/gold/`

## 0. Setup

Adds the `scripts/` folder to the Python path so we can import each layer's functions directly, instead of running them as separate subprocesses.

In [1]:
import sys
from pathlib import Path

# Project root is one level up from this notebook (notebooks/ -> project root)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SCRIPTS_DIR = PROJECT_ROOT / "scripts"

sys.path.insert(0, str(SCRIPTS_DIR))

print("Project root:", PROJECT_ROOT)
print("Scripts folder:", SCRIPTS_DIR)
print("Scripts found:", [f.name for f in SCRIPTS_DIR.glob('*.py')])

Project root: c:\Users\USER\MS2-finmark-data-pipeline
Scripts folder: c:\Users\USER\MS2-finmark-data-pipeline\scripts
Scripts found: ['bronze_layer.py', 'gold_layer.py', 'silver_layer.py']


## 1. Bronze Layer — Profile Raw Data

Profiles the three untouched raw datasets exactly as they arrived from the source system, no cleaning is applied. For each file, this checks row/column counts, missing values, duplicate rows, and undocumented junk columns. This is the baseline report that justifies every cleaning decision made in the Silver layer.

In [4]:
import profile_bronze_layer
import importlib
importlib.reload(profile_bronze_layer)

bronze_summary = profile_bronze_layer.profile_all()


PROFILING: event_logs.csv

Shape: 2000 rows x 50 columns

Meaningful columns (5): ['user_id', 'event_type', 'event_time', 'product_id', 'amount']
Junk/unnamed columns (45)

Data types of meaningful columns:
user_id        object
event_type     object
event_time     object
product_id     object
amount        float64

Missing values in meaningful columns:
        missing_count  missing_pct
amount           1016         50.8

Fully duplicated rows: 0

Checkout events: 260
Checkout events with NULL amount: 142
Percentage of checkouts missing amount: 54.6%

Event type counts:
event_type
login             276
checkout          260
wishlist_add      260
profile_update    255
add_to_cart       252
search            251
page_view         233
logout            213

PROFILING: marketing_summary.csv

Shape: 100 rows x 50 columns

Meaningful columns (5): ['date', 'users_active', 'total_sales', 'new_customers', 'report_generated']
Junk/unnamed columns (45)

Data types of meaningful columns:
date   

## 2. Silver Layer — Clean and Validate

Applies the actual cleaning rules: drops undocumented junk columns, removes exact duplicate rows, standardizes data types and text formatting, and flags (without deleting) rows with data quality issues such as checkout events with no recorded amount. Every cleaning decision is logged to `silver_validation_report.csv` (pass/fail checks) and `silver_quality_issues.csv` (counts of specific issues found and how they were handled).

In [5]:
import clean_silver_layer
importlib.reload(clean_silver_layer)

events, marketing, trends = clean_silver_layer.clean_all()

print("\nevent_logs_clean      :", events.shape)
print("marketing_summary_clean:", marketing.shape)
print("trend_report_clean     :", trends.shape)

Silver Layer cleaning completed. Outputs saved to data/silver/.

event_logs_clean      : (2000, 10)
marketing_summary_clean: (100, 7)
trend_report_clean     : (20, 4)


In [6]:
# Quick look at Silver validation and issue logs
import pandas as pd

validation = pd.read_csv(clean_silver_layer.SILVER_DIR / "silver_validation_report.csv")
issues = pd.read_csv(clean_silver_layer.SILVER_DIR / "silver_quality_issues.csv")

display(validation)
display(issues)

,file,check,status,details
0,event_logs.csv,Dropped undocumented junk columns,PASS,Dropped 45 columns
1,event_logs.csv,Allowed event_type values,PASS,0 invalid values
2,marketing_summary.csv,Dropped undocumented junk columns,PASS,Dropped 45 columns
3,marketing_summary.csv,Required field: date,PASS,0 missing/invalid values
4,marketing_summary.csv,Required field: users_active,PASS,0 missing/invalid values
5,marketing_summary.csv,Required field: total_sales,PASS,0 missing/invalid values
6,marketing_summary.csv,Required field: new_customers,PASS,0 missing/invalid values
7,marketing_summary.csv,Required field: report_generated,PASS,0 missing/invalid values
8,trend_report.csv,Dropped undocumented junk columns,PASS,Dropped 47 columns
9,trend_report.csv,Required field: week,PASS,0 missing/invalid values


,file,issue_type,count,handling
0,event_logs.csv,Exact duplicate rows,0,Removed before Silver export
1,event_logs.csv,Checkout rows with missing amount,142,Kept rows and flagged
2,event_logs.csv,Non-checkout rows with amount present,866,Kept rows and flagged
3,marketing_summary.csv,Exact duplicate rows,0,Removed before Silver export
4,trend_report.csv,Exact duplicate rows,0,Removed before Silver export


## 3. Gold Layer — Build Dashboard Tables

Transforms the cleaned data into business-ready summary tables, each built to feed a specific dashboard: top-level KPIs, the customer conversion funnel, feature usage by hour, hourly system load, daily checkout health, and a data quality/compliance summary. These are the final outputs that the Tableau dashboards will read from.

In [7]:
import build_gold_layer
importlib.reload(build_gold_layer)

gold_tables = build_gold_layer.build_all()

event_logs_clean     : (2000, 10)
marketing_summary    : (100, 7)
trend_report_clean   : (20, 4)
silver_quality_issues: (5, 4)
silver_validation    : (13, 4)
Saved: kpi_master_summary.csv
Saved: funnel_conversion.csv
Saved: product_feature_by_hour.csv
Saved: ops_hourly_load.csv
Saved: ops_checkout_health.csv
Saved: compliance_quality_summary.csv

Gold Layer build complete. 6 files saved to data/gold/.


In [8]:
# Preview each Gold table
for name, df in gold_tables.items():
    print(f"\n{'='*60}\n{name}\n{'='*60}")
    display(df.head())


kpi_master_summary


,metric,value,unit,source
0,Total Revenue (All Days),5767580.45,USD,marketing_summary
1,Average Daily Revenue,57675.80,USD/day,marketing_summary
2,Peak Daily Revenue,89585.41,USD,marketing_summary
3,Lowest Daily Revenue,20780.11,USD,marketing_summary
4,Average Daily Active Users,273.30,users,marketing_summary



funnel_conversion


,stage_order,stage_name,event_type,event_count,drop_off_from_previous,conversion_rate_pct
0,Stage 1,Product View,page_view,233,0,100.0
1,Stage 2,Search,search,251,0,107.7
2,Stage 3,Add to Cart,add_to_cart,252,0,108.2
3,Stage 4,Wishlist Add,wishlist_add,260,0,111.6
4,Stage 5,Checkout,checkout,260,0,111.6



product_feature_by_hour


...

,event_hour,total_events,checkout_events,unique_users,checkout_rate_pct,load_category
0,0,67,6,64,8.96,Medium
1,1,58,8,55,13.79,Low
2,2,97,10,88,10.31,High
3,3,78,16,73,20.51,Medium
4,4,66,10,60,15.15,Medium



ops_checkout_health


,event_date,total_checkouts,missing_amount,captured_revenue,failed_amount_rate_pct,status
0,2023-06-01,38,23,24554.08,60.53,Warning
1,2023-06-02,46,29,31115.79,63.04,Warning
2,2023-06-03,52,27,37833.91,51.92,Warning
3,2023-06-04,43,22,34202.65,51.16,Warning
4,2023-06-05,39,21,28487.36,53.85,Warning



compliance_quality_summary


,metric,value,status
0,Total Silver event records,2000,INFO
1,Duplicate rows removed,0,PASS
2,Silver validation checks passed,13,PASS
3,Silver validation checks failed,0,INFO
4,Missing amount fields (all events),1016,WARNING


## 4. Pipeline Summary

Confirms that every expected output file from all three layers was actually created and reports OK/MISSING for each one. This is run last to catch a silent failure such as if a script ran without errors but, for some reason, didn't write its file.

In [9]:
expected_outputs = {
    "Bronze": [PROJECT_ROOT / "data" / "bronze" / "bronze_quality_report.csv"],
    "Silver": [
        PROJECT_ROOT / "data" / "silver" / "event_logs_clean.csv",
        PROJECT_ROOT / "data" / "silver" / "marketing_summary_clean.csv",
        PROJECT_ROOT / "data" / "silver" / "trend_report_clean.csv",
        PROJECT_ROOT / "data" / "silver" / "silver_validation_report.csv",
        PROJECT_ROOT / "data" / "silver" / "silver_quality_issues.csv",
    ],
    "Gold": [
        PROJECT_ROOT / "data" / "gold" / "kpi_master_summary.csv",
        PROJECT_ROOT / "data" / "gold" / "funnel_conversion.csv",
        PROJECT_ROOT / "data" / "gold" / "product_feature_by_hour.csv",
        PROJECT_ROOT / "data" / "gold" / "ops_hourly_load.csv",
        PROJECT_ROOT / "data" / "gold" / "ops_checkout_health.csv",
        PROJECT_ROOT / "data" / "gold" / "compliance_quality_summary.csv",
    ],
}

print("Pipeline Run Summary\n" + "="*60)
for layer, files in expected_outputs.items():
    print(f"\n{layer} Layer:")
    for f in files:
        status = "OK" if f.exists() else "MISSING"
        print(f"  [{status}] {f.relative_to(PROJECT_ROOT)}")

Pipeline Run Summary

Bronze Layer:
  [OK] data\bronze\bronze_quality_report.csv

Silver Layer:
  [OK] data\silver\event_logs_clean.csv
  [OK] data\silver\marketing_summary_clean.csv
  [OK] data\silver\trend_report_clean.csv
  [OK] data\silver\silver_validation_report.csv
  [OK] data\silver\silver_quality_issues.csv

Gold Layer:
  [OK] data\gold\kpi_master_summary.csv
  [OK] data\gold\funnel_conversion.csv
  [OK] data\gold\product_feature_by_hour.csv
  [OK] data\gold\ops_hourly_load.csv
  [OK] data\gold\ops_checkout_health.csv
  [OK] data\gold\compliance_quality_summary.csv
